# Explore Scoring Model Behavior

Requirements: `pip install transformers supabase python-dotenv`

You should already have torch and pandas in your environment.

In [1]:
import os
import ipywidgets as widgets
from IPython.display import HTML

import torch
import numpy as np
import pandas as pd
from transformers import pipeline
from supabase import create_client, Client
from dotenv import load_dotenv
from tqdm.auto import tqdm

from models import ModernBERT_v2, ModelInput

load_dotenv()
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")
supabase = create_client(supabase_url=url, supabase_key=key)

In [2]:
# Get chunk data
chunk_df = pd.read_csv(
    "/home/jovyan/active-projects/shared-resources/strapi-exports/data/strapi_data.csv",
    index_col=0,
)

# There are some duplicate chunks. Likely multiple pages linked to the same chunk.
chunk_df = chunk_df.loc[~chunk_df.index.duplicated(keep="first"), :]
chunk_df.index.value_counts()

chunk_slug
Discussion-II-5852                                       1
Structure-and-Function-of-the-Respiratory-System-6247    1
The-Conducting-Airways-6248                              1
The-Respiratory-Airways-6249                             1
Gas-Exchange-6250                                        1
                                                        ..
16-Training-Records-2031                                 1
17-Training-Program-Evaluation-2032                      1
31-Job-Training-and-Transfer-Training-2023               1
33-Refresher-Training-2024                               1
41-Requirements-and-Placement-2039                       1
Name: count, Length: 585, dtype: int64

In [6]:
# Get logs data
client_names = ["hinze_rmp_summer_2025", "middle_georgia"]

logs_response = (
    supabase.table("logs")
    .select("*")
    # .in_("client_name", client_names)  # Optionally filter by client_name
    .eq("api_endpoint", "/score/answer")  # CRI Endpoint
    .order("created_at", desc=True)  # Optionally sort by most recent API calls
    .limit(10_000)  # Number of samples to collect
    .execute()
)

logs_df = pd.json_normalize(logs_response.data)

# Join API query data with content data extracted from Strapi (to get volume slug, page slug, chunk text, question, reference answer)
df = logs_df.merge(
    chunk_df, left_on="request_body.chunk_slug", right_on="chunk_slug", how="left"
)

# Construct DataFrame
columns_to_keep = [
    "volume_slug",
    "request_body.page_slug",
    "request_body.chunk_slug",
    "chunk_text",
    "question",
    "answer",
    "request_body.answer",
    "response_body.level",
    "response_body.score",
    "response_body.feedback",
    "response_body.is_passing",
]

df = df[columns_to_keep]
df = df.rename(
    columns={
        "answer": "reference",
        "request_body.answer": "candidate",
        "chunk_text": "text",
    }
)

# Clean up column names
df.columns = [
    col.replace("request_body.", "").replace("response_body.", "") for col in df.columns
]

# Remove rows without necessary information
df = df[~df["question"].isna() & ~df["text"].isna() & ~df["reference"].isna()]  # Some rows have NaN values

# Random sample
df = df.sample(500)

,volume_slug,page_slug,chunk_slug,text,question,reference,candidate,level,score,feedback,is_passing
0,research-methods-in-psychology-demo,21-practical-strategies-for-psychological-meas...,Implementing-the-Measure-1684t,You will want to implement any measure in a wa...,How can you minimize participant reactivity in...,"You can make the procedure clear and brief, gu...",You can maximize participant reactivity in a r...,4.0,3.829048,Excellent job! Your response demonstrates deep...,True
1,research-methods-in-psychology-demo,21-practical-strategies-for-psychological-meas...,Conceptually-Defining-the-Construct-1681t,Having a clear and complete conceptual definit...,Why is having a clear and complete conceptual ...,It allows you to make sound decisions about ex...,Having a clear and complete conceptual definit...,4.0,3.861891,Excellent job! Your response demonstrates deep...,True
2,research-methods-in-psychology-demo,21-practical-strategies-for-psychological-meas...,Implementing-the-Measure-1684t,You will want to implement any measure in a wa...,How can you minimize participant reactivity in...,"You can make the procedure clear and brief, gu...","Procedures can be clear and brief, guarantee a...",4.0,3.666522,Excellent job! Your response demonstrates deep...,True
3,research-methods-in-psychology-demo,21-practical-strategies-for-psychological-meas...,Implementing-the-Measure-1684t,You will want to implement any measure in a wa...,How can you minimize participant reactivity in...,"You can make the procedure clear and brief, gu...",One way is to make the study as clear and brie...,2.0,1.904170,"You are on the right track, but your response ...",False
4,research-methods-in-psychology-demo,21-practical-strategies-for-psychological-meas...,Conceptually-Defining-the-Construct-1681t,Having a clear and complete conceptual definit...,Why is having a clear and complete conceptual ...,It allows you to make sound decisions about ex...,It is important to have a clear and complete c...,4.0,3.580584,Excellent job! Your response demonstrates deep...,True
...,...,...,...,...,...,...,...,...,...,...,...
9995,communication-for-business-success,19-2-group-life-cycles-and-member-roles,Life-Cycle-of-Member-Roles-2365,"Over time and projects, you gradually increase...",What role does a new group member play in a gr...,A new group member learns the group's rules an...,ef,NaN,0.000000,NaN,False
9996,communication-for-business-success,19-2-group-life-cycles-and-member-roles,Group-Life-Cycle-Patterns-2364,"Adjourning Stage\n\nNow, as typically happens,...",What are the stages of group development accor...,The stages of group development according to T...,"Forming, storming, norming, and adjourning.",NaN,2.000000,NaN,True
9997,communication-for-business-success,19-1-what-is-a-group,Relationships-and-Group-Dynamics-2356,"Relationships are part of any group, and can b...",What are some common elements that describe re...,"Common elements include shared challenges, col...",ewf,NaN,0.000000,NaN,False
9998,communication-for-business-success,19-1-what-is-a-group,Types-of-Groups-in-the-Workplace-2355,"As a skilled business communicator, learning m...",What types of groups can individuals within a ...,Individuals within a business or organization ...,no idea,NaN,0.000000,NaN,False


## Re-score the data

In [7]:
mb_w_reference_answer_path = "/home/jovyan/active-projects/itell-question-generation/results/modernbert_multirc_reading_engagement"

MBV2 = ModernBERT_v2(mb_w_reference_answer_path)

Device set to use cuda


In [9]:
def score(df, pipe):
    all_preds = []

    for ex in tqdm(df.to_dict("records")):
        model_input = ModelInput.from_dict(ex)
        pred = pipe(model_input)
        all_preds.append(pred)

    return all_preds

df = df.sample(500)
df["new_score"] = score(df, MBV2)

  0%|          | 0/500 [00:00<?, ?it/s]

In [11]:
# Feel free to modify filters

PASSING_THRESHOLD = 2.3  # Scores >= this value are passing

filters = (
    (abs(df["score"] - df["new_score"]) > 1.0)  # scores differ by more than 1.0
    | ((df["score"] > PASSING_THRESHOLD) & (df["new_score"] < PASSING_THRESHOLD))  # scores on different sides of threshold
    | ((df["score"] < PASSING_THRESHOLD) & (df["new_score"] > PASSING_THRESHOLD))  # scores on different sides of threshold
) 

# For example, this will select scores that are within TOLERANCE of TARGET_SCORE
TARGET_SCORE = 2.3
TOLERANCE = 0.2
filters = (df["new_score"] - TARGET_SCORE).abs() < TOLERANCE

# Apply filters
df_filtered = df[filters]

df_filtered

,volume_slug,page_slug,chunk_slug,text,question,reference,candidate,level,score,feedback,is_passing,new_score
5721,introduction-to-computing,3-1-control-structures,Exception-Handling-2864,"Earlier in our material, we covered the idea o...",Can we use errors in the design of our programs?,"Yes, exception handling allows us to anticipat...",Yes we can use errors to ensure that the progr...,NaN,2.0,NaN,True,2.455670
6716,introduction-to-computing,3-4-functions,A-Function-with-a-Parameter-3420,The function from Figure 3.4.5 just returns th...,What does the function returnYenAmount(10) out...,¥10,¥ symbol then 10 is returned but there is not ...,NaN,2.0,NaN,True,2.350254
2016,introduction-to-computing,4-2-strings,String-Concatenation-3145,String concatenation means putting multiple st...,What does the + operator do in string concaten...,The + operator tacks each string on to the pre...,It puts the strings together / adds them toget...,NaN,2.0,NaN,True,2.392624
5586,introduction-to-computing,3-5-error-handling,Catching-a-Specific-Error-3077,"Note that the way we’ve written Figure 3.5.3, ...",What happens when a ValueError occurs in the c...,The code jumps to the except block specificall...,it prints error then Done!,NaN,2.0,NaN,True,2.232547
5838,introduction-to-computing,3-1-control-structures,The-Dangers-of-Scope-in-Python-2873,Scope in Python also presents a danger. Take a...,What happens when line 6 is never run in the c...,An error occurs on line 8 because the variable...,It gives us an error,NaN,2.0,NaN,True,2.484062
6843,introduction-to-computing,3-2-conditionals,If-Then-Else-If-Else-2916,Now let’s throw our else-if statements into th...,What is the purpose of using 'elif' in Python ...,The 'elif' keyword is used to create an else-i...,Allow to check multiple conditions in a struct...,NaN,0.0,NaN,False,2.424858
5178,introduction-to-computing,3-4-functions,Parameter-Mismatch-3422,"When we define a function, part of that defini...",What happens if we don’t supply arguments for ...,The program will throw a TypeError indicating ...,an error will occur,NaN,2.0,NaN,True,2.486799
5278,introduction-to-computing,3-3-loops,For-Loops-3021,A for loop repeats some code a certain number ...,What is a for-each loop and how does it differ...,A for-each loop simplifies iterating over item...,a for-each loop is applicable to every unit in...,NaN,2.0,NaN,True,2.316450
5114,introduction-to-computing,3-2-conditionals,If-Then-Else-2904,"Right now, the code just checks if it’s rainin...",How do we add a recommendation for t-shirt and...,We add an else block with the corresponding li...,"by adding if todaysWeather== ""not raining""",NaN,1.0,NaN,False,2.268202
3959,introduction-to-computing,4-5-dictionaries,Simple-Examples-of-Dictionaries-3247,"First, let’s see some code that counts the wor...",What is the purpose of the code snippet provid...,To count the occurrences of each word in a giv...,it shows how many times each word in the strin...,NaN,2.0,NaN,True,2.375541


In [12]:
current_idx = 0

def create_display(df, idx):
    """Create the display for a given row index"""
    if idx < 0 or idx >= len(df):
        return None

    row = df.iloc[idx]

    # Round scores to 2 decimal places
    score = round(row["score"], 2)
    new_score = round(row["new_score"], 2)

    # Determine passing status and styling
    score_passing = score >= PASSING_THRESHOLD
    new_score_passing = new_score >= PASSING_THRESHOLD

    score_bg = "#c8e6c9" if score_passing else "#ffcdd2"
    new_score_bg = "#c8e6c9" if new_score_passing else "#ffcdd2"

    html_content = f"""
    <div style="font-family: Arial, sans-serif; line-height: 1.6;">
        <div style="background-color: #f0f0f0; padding: 10px; border-radius: 5px; margin-bottom: 15px;">
            <strong>Row {idx + 1} of {len(df)}</strong>
        </div>
        
        <div style="background-color: #e8f4f8; padding: 12px; border-left: 4px solid #2196F3; margin-bottom: 15px;">
            <strong>Question:</strong><br>
            {row["question"]}
        </div>
        
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;">
            <div style="background-color: {score_bg}; padding: 10px; border-radius: 8px; border: 2px solid #388e3c; text-align: center;">
                <div style="font-size: 12px; color: #333; margin-bottom: 8px;">Original Score</div>
                <div style="font-size: 18px; font-weight: bold; color: #1b5e20;">{score}</div>
            </div>
            
            <div style="background-color: {new_score_bg}; padding: 10px; border-radius: 8px; border: 2px solid #388e3c; text-align: center;">
                <div style="font-size: 12px; color: #333; margin-bottom: 8px;">New Score</div>
                <div style="font-size: 18px; font-weight: bold; color: #1b5e20;">{new_score}</div>
            </div>
        </div>
                
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div style="background-color: #e8f5e9; padding: 15px; border: 2px solid #4CAF50; border-radius: 5px;">
                <h3 style="margin-top: 0; color: #2e7d32;">Reference Answer</h3>
                <div style="font-size: 14px; line-height: 1.8;">
                    {row["reference"]}
                </div>
            </div>
            
            <div style="background-color: #fce4ec; padding: 15px; border: 2px solid #E91E63; border-radius: 5px;">
                <h3 style="margin-top: 0; color: #880e4f;">Student Response</h3>
                <div style="font-size: 14px; line-height: 1.8;">
                    {row["candidate"]}
                </div>
            </div>
        </div>

        <hr style="margin: 20px 0;">

        <div style="background-color: #fff3e0; padding: 12px; border-left: 4px solid #FF9800; margin-bottom: 15px;">
            <strong>Text/Context:</strong><br>
            {row["text"]}
        </div>

    </div>
    """

    return html_content


# Create output widget
output = widgets.Output()

# Create buttons
prev_button = widgets.Button(description="← Previous", button_style="info")
next_button = widgets.Button(description="Next →", button_style="info")
status_label = widgets.Label(value=f"Row 1 of {len(df_filtered)}")


def on_prev_clicked(b):
    global current_idx
    current_idx = max(0, current_idx - 1)
    update_display()


def on_next_clicked(b):
    global current_idx
    current_idx = min(len(df_filtered) - 1, current_idx + 1)
    update_display()


def update_display():
    output.clear_output(wait=True)
    html_content = create_display(df_filtered, current_idx)
    status_label.value = f"Row {current_idx + 1} of {len(df_filtered)}"
    with output:
        display(HTML(html_content))


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)

# Create layout
button_box = widgets.HBox([prev_button, status_label, next_button])
layout = widgets.VBox([button_box, output])

# Initial display
update_display()

# Show the interface
display(layout)

LH - I am thinking the following for thresholds:
1. (-inf, 1.5)
2. [1.5, 2.1)
3. [2.1, 3.3)
4. [3.3, +inf)